# Flow Matching in Audio Enhancement - Tutorial

Authors: Maja Byrecka, Sonia Weiss

## Overview

In this tutorial, we explore Flow Matching as a method for audio enhancement, focusing on noise reduction.

**Speech enhancement** is the task of taking a degraded recording (e.g. speech with background noise) and producing a cleaner version of it, ideally one that sounds as if it had been recorded in a quiet room. It is used, for example, in hearing aids, video calls and as a preprocessing step for speech recognition.

We go through the topic step by step:

1. **Theoretical background**: how audio is represented in a computer, what generative models are, and how Flow Matching differs from diffusion models.
2. **Data**: a short clean speech recording and its noisy version. We listen to both and compare their mel-spectrograms.
3. **Pretrained model**: we run **FlowSE**, a Flow Matching model for speech enhancement, on the noisy sample and compare the result with the input.
4. **Flow Matching in detail**: what happens inside the model.
5. **Implementation from scratch**: we implement a small conditional Flow Matching model in PyTorch.
6. **Summary**.

After this tutorial you should be able to read a mel-spectrogram, explain the core idea of Flow Matching, and say how it differs from diffusion.

## Theoretical Background

### How audio is represented

Sound is a change of air pressure over time. A microphone turns it into an electrical signal, which is then **sampled**: the signal value (amplitude) is measured many times per second. The number of measurements per second is the **sample rate**. In this tutorial it is 22,050 Hz, so one second of audio is 22,050 numbers. A WAV file is essentially a long 1D array of such samples. A 10-second recording is already about 220,000 values.

This raw form is called a **waveform**. It is hard to interpret directly, because the important properties of speech (pitch, vowels, consonants) are about **frequencies**, not individual sample values. That is why we usually move to a **time-frequency representation**:

- **Spectrogram**: the signal is split into short, overlapping windows, and each window is transformed with the Fourier transform (STFT). The result is a 2D "image": time on the x-axis, frequency on the y-axis, energy (in dB) as color.
- **Mel-spectrogram**: a spectrogram whose frequencies are grouped into a few bands (here `N_MELS = 80`) according to the **mel scale**. The mel scale mimics human hearing: we tell low frequencies apart much better than high ones. The result is compact and keeps what matters for perception.

On a mel-spectrogram, speech appears as horizontal stripes (harmonics of the voice) and dark gaps (pauses). Noise looks different. **White noise**, for example, spreads its energy evenly over all frequencies and covers the whole image like fog.

Many modern speech models (text-to-speech, enhancement) operate on mel-spectrograms and treat them much like images. A mel-spectrogram does not store the **phase** of the signal, so it cannot be turned back into audio by a simple formula. For this we need a **vocoder**, a separate neural network that generates a waveform from a mel-spectrogram. FlowSE uses the Vocos vocoder.

**Libraries for audio:** we use `librosa` to load audio and compute (mel-)spectrograms, `soundfile` to read and write WAV files, `librosa.display` with `matplotlib` to plot spectrograms, and `IPython.display.Audio` to listen to recordings directly in the notebook.

### Enhancement as a generative problem

Classical denoising methods (e.g. spectral subtraction, Wiener filtering) estimate the noise and subtract it. Many neural approaches are **discriminative**: the network learns a direct mapping "noisy in → clean out", e.g. by predicting a mask over the spectrogram. This works well, but under strong noise the output often becomes over-smoothed or contains artifacts, because the network averages over all possible clean signals.

**Generative** approaches instead learn the *distribution* of clean speech, conditioned on the noisy input, and then *sample* a realistic clean signal from it. The two most popular families of generative models today are **diffusion models** and **Flow Matching**.

### Diffusion models

The idea behind diffusion models is:

1. **Forward process (destroying data):** take a clean example and add a little Gaussian noise to it, many times in a row, until nothing is left but pure noise.
2. **Reverse process (learned):** train a neural network that, given a noisy example and the information about how noisy it is (the time step `t`), predicts the noise that was added. Removing that noise moves the example one small step back towards clean data.
3. **Generation:** start from pure random noise and apply the network many times (typically tens to hundreds of steps), removing a bit of noise each time.

**Stable Diffusion** is probably the best-known example. It is a text-to-image model that runs this process not on pixels but in a compressed *latent* space, guided by a text prompt. The same principle has been applied to audio, e.g. in diffusion-based speech enhancement models such as SGMSE+.

The main drawback is that the path from noise to data is **curved and stochastic** (mathematically, a stochastic differential equation), so generation needs many network evaluations and is slow.

### Flow Matching

Flow Matching (Lipman et al., 2022) asks a simpler question: *if we want to move a sample from a source distribution to the data distribution, which direction should it move in at each moment?*

- Take a **source** sample $x_0$ (e.g. Gaussian noise) and a **target** sample $x_1$ (e.g. clean speech).
- Connect them with a **straight line**: $x_t = (1 - t)\,x_0 + t\,x_1$ for $t \in [0, 1]$.
- Along this line the velocity is constant: $u = x_1 - x_0$.
- Train a network $v_\theta(x_t, t)$ to predict this velocity. The loss is a plain regression: $\mathcal{L} = \lVert v_\theta(x_t, t) - (x_1 - x_0) \rVert^2$.

The trained network defines a **vector field**: for every point and every time it says where to go next (think of a map of arrows, or a GPS giving directions). To generate, we start at $x_0$ and follow the arrows, i.e. solve an ordinary differential equation (ODE) $\frac{dx}{dt} = v_\theta(x, t)$. The simplest solver is the Euler method: repeat $x \leftarrow x + \Delta t \cdot v_\theta(x, t)$.

For enhancement the model is **conditional**: the network additionally receives the noisy recording (and, in FlowSE, also the transcript), so the arrows lead towards the clean version of *this particular* recording rather than to any random speech.

### Diffusion vs Flow Matching

| | Diffusion | Flow Matching |
|---|---|---|
| Core idea | learn to undo gradual noising | learn a velocity field that moves samples along a chosen path |
| What the network predicts | the added noise (or the score) | the velocity $x_1 - x_0$ |
| Path from source to data | curved, stochastic (SDE) | straight (or nearly), deterministic (ODE) |
| Sampling steps | typically tens to hundreds | typically a few to a few dozen |
| Source distribution | Gaussian noise | any distribution (noise, or even noisy speech itself) |
| Training | simple, but tied to a noise schedule | simple regression, few design choices |

The two families are closely related: diffusion can be seen as Flow Matching with a particular, curved path. Flow Matching's straighter paths are the main reason it generates high-quality samples in fewer steps, which matters for audio, where we want fast (ideally real-time) processing.

## Libraries

We use standard Python modules (`pathlib`, `json`, `os`, `shutil`, `subprocess`, `sys`, `time`) for paths, files and running external scripts, plus the following libraries:

**Audio processing libraries**
- `librosa` — audio analysis (loading, feature extraction, spectrograms).
- `librosa.display` — plotting audio features (waveforms, spectrograms).
- `soundfile` (`sf`) — read/write audio files (WAV, FLAC).

**Numerics & plotting**
- `numpy` (`np`) — array math and numerical operations.
- `matplotlib.pyplot` (`plt`) — plotting and visualization.

**Config**
- `yaml` — read/write YAML config files.

**Pretrained models**
- `huggingface_hub` (`hf_hub_download`) — download pretrained model files (FlowSE checkpoint, Vocos vocoder) from the Hugging Face Hub.

**Notebook display**
- `IPython.display` (`Audio`, `Markdown`, `display`) — render audio players, Markdown, and rich output inline in Jupyter.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import yaml

from huggingface_hub import hf_hub_download
from IPython.display import Audio, Markdown, display

## Data

The recordings come from the **LJ Speech Dataset** (Keith Ito, 2017), a public-domain dataset widely used in speech research, especially for text-to-speech. It contains about 13,100 short clips (around 24 hours in total) of a single female speaker reading passages from non-fiction books. Every clip has a text transcript and is stored as a mono, 16-bit WAV file at 22,050 Hz.

We use a single clip: the first one in the dataset (`LJ001-0001`). Its transcript is stored in the `TRANSCRIPT` variable below and is later passed to FlowSE as additional conditioning:

> *"Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition"*

The `data/` folder contains two files:

| File | Content |
|---|---|
| `lj_clean.wav` | the original clean recording |
| `lj_noisy.wav` | the same recording with artificially added noise |

Both files are mono, 16-bit PCM, 22,050 Hz, about 9.66 s long, and aligned sample by sample.

The noisy version was created by **adding** noise to the clean signal: `noisy = clean + noise`. The noise is **white Gaussian noise**: random values with the same energy at every frequency and the same level throughout the whole recording. Its level corresponds to a **signal-to-noise ratio (SNR) of 10 dB**, meaning the speech has 10 times more power than the noise. The speech is still clearly understandable, but the hiss is very noticeable.

This clean + noisy pair setup is also how speech enhancement models are usually trained: clean recordings are mixed with different noises at different SNRs, and the model learns to recover the clean signal. Having the clean reference lets us check how close the enhanced output is to the original.

In [ ]:
# Input audio files
DATA_DIR = Path("data")
CLEAN_PATH = DATA_DIR / "lj_clean.wav"
NOISY_PATH = DATA_DIR / "lj_noisy.wav"

# Audio representation parameters
SAMPLE_RATE = 22_050
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 80

# External model configuration
FLOWSE_REPO = Path("FlowSE").resolve()
MODEL_DIR = Path("models").resolve()

CHECKPOINT_REPO_ID = "flowse/wenetspeech4tts_Premium.pt.tar"
CHECKPOINT_FILENAME = "wenetspeech4tts_Premium.pt.tar"

VOCODER_REPO_ID = "charactr/vocos-mel-24khz"
VOCODER_FILENAMES = ("config.yaml", "pytorch_model.bin")

# Inference configuration
EXAMPLE_ID = "lj_noisy"
TRANSCRIPT = (
    "Printing, in the only sense with which we are at present "
    "concerned, differs from most if not from all the arts "
    "and crafts represented in the Exhibition"
)

# CPU resource limits
CPU_THREADS = 2

## Audio Representations

A WAV file stores audio as discrete samples. Time-frequency representations are commonly used for audio processing. Most typical formats are:
- Spectrograms - representing the magnitude of Short-time Fourier Transform (STFT) components over time;
- Mel-spectrograms - calculated as spectrograms transformed onto the mel scale, providing a compact representation corresponding to human sound perception.

In this tutorial, we will use mel-spectrograms as the input and output representations.

### Parameters

The mel-spectrograms are computed with the parameters defined above:

| Parameter | Value | Meaning and value |
|---|---|---|
| `SAMPLE_RATE` | 22,050 Hz | Samples per second. Native rate of LJ Speech, so no resampling is needed. |
| `N_FFT` | 1024 samples | Length of one STFT window (~46 ms). Short enough to follow changes in speech, long enough for good frequency resolution. |
| `HOP_LENGTH` | 256 samples | Shift between consecutive windows (~11.6 ms, 75% overlap). |
| `N_MELS` | 80 bands | Number of mel bands (rows of the mel-spectrogram). Standard value in speech models. |

This combination (22,050 Hz, 1024, 256, 80) is the de facto standard for LJ Speech in text-to-speech research.

In the code below we also use:
- `power=2.0`: energy (squared magnitude) instead of amplitude;
- `fmin=0`, `fmax=sample_rate/2`: the full frequency range;
- `power_to_db(..., top_db=80)`: conversion to decibels, with values more than 80 dB below the loudest point clipped, so very quiet regions do not dominate the color scale. Both spectrograms share the same reference value, so their colors can be compared directly.

In [ ]:
wav_clean, sample_rate = librosa.load(CLEAN_PATH)
wav_noisy, _ = librosa.load(NOISY_PATH)

In [ ]:
display(Markdown("Clean Speech: "))
display(Audio(wav_clean, rate=SAMPLE_RATE))

display(Markdown("Noisy Speech: "))
display(Audio(wav_noisy, rate=SAMPLE_RATE))

In [ ]:
mel_clean = librosa.feature.melspectrogram(y=wav_clean, sr=sample_rate, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS, fmin=0, fmax=sample_rate/2, power=2.0)
mel_noisy = librosa.feature.melspectrogram(y=wav_noisy, sr=sample_rate, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS, fmin=0, fmax=sample_rate/2, power=2.0)

reference = max(float(mel_clean.max()), float(mel_noisy.max()))

mel_clean_db = librosa.power_to_db(mel_clean, ref=reference, top_db=80)
mel_noisy_db = librosa.power_to_db(mel_noisy, ref=reference, top_db=80)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for ax, spectrogram, title in [
    (axes[0], mel_clean_db, "Clean Speech"),
    (axes[1], mel_noisy_db, "Speech with Added Noise"),
]:
    librosa.display.specshow(spectrogram, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel", vmin=-80, vmax=0, ax=ax)
    ax.set_title(title)
    ax.set_ylabel("Frequency (mel)")

axes[-1].set_xlabel("Time (seconds)")
plt.tight_layout()
plt.show()

print(f"Sample rate: {SAMPLE_RATE:,} Hz")
print(f"Clean duration: {len(wav_clean) / SAMPLE_RATE} s")
print(f"Noisy duration: {len(wav_noisy) / SAMPLE_RATE} s")

## Flow Matching Models

### Architecture high-level description


### FlowSE

One of the state-of-the-art flow matching architectures designed for audio enhancement task is FlowSE. 

FlowSE is a Flow Matching-based generative model designed for speech enhancement. It learns a time-dependent vector field that transforms noisy speech representations into clean speech representations by solving an ordinary differential equation (ODE). Unlike traditional denoising approaches, it models a continuous transformation from noisy to clean speech through iterative refinement.

Let's run the inference and analyze the results.

In [ ]:
def setup_flowse(
    *,
    repo_dir: Path = FLOWSE_REPO,
    model_dir: Path = MODEL_DIR,
) -> dict:
    """
    Clone FlowSE, download pretrained checkpoints and prepare CPU inference config
    """
    repo_dir = repo_dir.resolve()
    model_dir = model_dir.resolve()

    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/Honee-W/FlowSE.git", str(repo_dir)], check=True)

    base_config_path = repo_dir / "config" / "train.yaml"
    model_dir.mkdir(parents=True, exist_ok=True)

    # Download checkpoints
    print("Downloading models...")
    checkpoint_path = Path(hf_hub_download(repo_id=CHECKPOINT_REPO_ID, filename=CHECKPOINT_FILENAME, local_dir=str(model_dir))).resolve()

    vocoder_dir = (repo_dir / "vocos-mel-24khz").resolve()
    vocoder_dir.mkdir(parents=True, exist_ok=True)

    for filename in VOCODER_FILENAMES:
        hf_hub_download(repo_id=VOCODER_REPO_ID, filename=filename, local_dir=str(vocoder_dir))

    manifest_path = repo_dir / "datalist" / "tutorial" / "text.json"

    output_dir = (repo_dir / "test_out").resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    # Add CPU inference configuration
    config = yaml.safe_load(base_config_path.read_text(encoding="utf-8"))
    infer = config["infer"]

    infer["test"]["use_cuda"] = False
    infer["test"]["checkpoint"] = str(checkpoint_path.parent)
    infer["test"]["pt_name"] = checkpoint_path.name
    infer["datareader"]["mix_json"] = str(manifest_path.resolve())

    infer["save"]["dir"] = str(output_dir)

    vocoder_config = infer["nnet_conf"]["vocoder"]
    vocoder_config["is_local"] = True
    vocoder_config["local_path"] = str(vocoder_dir)

    cpu_config_path = (repo_dir / "config" / "tutorial_cpu.yaml").resolve()
    cpu_config_path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")

    return {
        "repo_dir": repo_dir,
        "config_path": cpu_config_path,
        "manifest_path": manifest_path,
        "checkpoint_path": checkpoint_path,
        "output_dir": output_dir,
    }

In [ ]:
flowse = setup_flowse()

In [ ]:
def run_flowse_inference(
    noisy_audio: np.ndarray,
    sample_rate: int,
    flowse: dict,
    *,
    cpu_threads: int = CPU_THREADS,
) -> dict:
    """
    Run pretrained FlowSE speech enhancement on a single noisy recording
    """
    repo_dir = flowse["repo_dir"]
    manifest_path = flowse["manifest_path"]

    example_dir = repo_dir / "datalist" / "tutorial" / "noisy"

    example_dir.mkdir(parents=True, exist_ok=True)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)

    tutorial_input_path = example_dir / f"{EXAMPLE_ID}.wav"

    sf.write(tutorial_input_path, noisy_audio, sample_rate, subtype="PCM_16")
    manifest_path.write_text(json.dumps({EXAMPLE_ID: TRANSCRIPT}, indent=2), encoding="utf-8")

    # Preserve the test input location expected by the FlowSE setup
    test_input_path = repo_dir / "datalist" / "test" / "noisy" / f"{EXAMPLE_ID}.wav"
    test_input_path.parent.mkdir(parents=True, exist_ok=True)

    sf.write(test_input_path, noisy_audio, sample_rate, subtype="PCM_16")

    enhanced_path = flowse["output_dir"] / f"{EXAMPLE_ID}.wav"

    # Run inference with bounded CPU thread usage
    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = str(cpu_threads)
    env["MKL_NUM_THREADS"] = str(cpu_threads)
    env["OPENBLAS_NUM_THREADS"] = str(cpu_threads)

    env["PYTHONIOENCODING"] = "utf-8"

    print("Inference...")
    result = subprocess.run(
        [sys.executable, str(repo_dir / "infer.py"), "-conf", str(flowse["config_path"])],
        cwd=repo_dir,
        env=env,
        capture_output=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(f"FlowSE inference failed:\n{result.stderr}")

    return {
        "enhanced_path": enhanced_path,
        "checkpoint_path": flowse["checkpoint_path"],
        "output_dir": flowse["output_dir"],
    }

In [ ]:
flowse_result = run_flowse_inference(noisy_audio=wav_noisy, sample_rate=SAMPLE_RATE, flowse=flowse)
enhanced_path = flowse_result["enhanced_path"]

print(f"Enhanced audio saved in: {enhanced_path}")

### Inspect the Enhanced Sample

The inference pipeline returns a WAV file, which can be loaded and inspected.

Most common ways used for comparing enhanced and raw samples are:
- Listening - allows to assess speech clarity, residual noise, and possible speech artifacts;
- Mel-spectrograms analysis - allows visual samples comparision;

Mel-spectrogram analysis is especially useful for accurate assessment of multiple samples, but reduction in visible energy does not necessarily imply better intelligibility and sound quality. Usually it is treated as a support and does not fullty replace the listening tests.

In [ ]:
enhanced, sr_enhanced = sf.read(enhanced_path, dtype="float32")

print(f"Output file: {enhanced_path}")
print(f"Output sample rate: {sr_enhanced:,} Hz")
print(f"Output duration: {len(enhanced) / sr_enhanced:.2f} s")

display(Markdown("Raw sample: "))
display(Audio(wav_noisy, rate=SAMPLE_RATE))

display(Markdown("Enhanced sample: "))
display(Audio(enhanced, rate=sr_enhanced))

In [ ]:
comparison_sr = sr_enhanced
comparison_n_fft = 2048

wav_noisy_comparison = librosa.resample(wav_noisy, orig_sr=SAMPLE_RATE, target_sr=comparison_sr)

mel_noisy_comparison = librosa.feature.melspectrogram(y=wav_noisy_comparison, sr=comparison_sr, n_fft=comparison_n_fft, hop_length=HOP_LENGTH, n_mels=N_MELS, fmin=0, fmax=comparison_sr / 2, power=2.0)
mel_enhanced = librosa.feature.melspectrogram(y=enhanced, sr=sr_enhanced, n_fft=comparison_n_fft, hop_length=HOP_LENGTH, n_mels=N_MELS, fmin=0, fmax=comparison_sr / 2, power=2.0)

comparison_reference = max(float(mel_noisy_comparison.max()), float(mel_enhanced.max()), 1e-10)

noisy_comparison_db = librosa.power_to_db(mel_noisy_comparison, ref=comparison_reference, top_db=80)

enhanced_comparison_db = librosa.power_to_db(mel_enhanced, ref=comparison_reference, top_db=80)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

for ax, spectrogram, title in [
    (axes[0], noisy_comparison_db, "Before Enhancement: Noisy Speech"),
    (axes[1],enhanced_comparison_db,"After Enhancement: FlowSE Output"),
]:
    librosa.display.specshow(spectrogram, sr=comparison_sr, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel", vmin=-80, vmax=0, ax=ax)
    ax.set_title(title)
    ax.set_ylabel("Frequency (mel)")

axes[-1].set_xlabel("Time (seconds)")
plt.tight_layout()
plt.show()

## Flow Matching Architecture - Details

## Flow Matching Low-Level Implementation

## Summary